In [1]:
import numpy as np
import inspect
import importlib
import Utils
importlib.reload(Utils)
from rasterstats import zonal_stats
import Constants
importlib.reload(Constants)
import ConstantObjects
importlib.reload(ConstantObjects)
import matplotlib.pyplot as plt
import os
import rasterio
from rasterio.warp import reproject, Resampling

print("Done with cell!")



[Line 14] n_cols in ConstantObjects: 21
[Line 16] n_cols: 21, n_rows: 18
[Line 27] n_cols in ConstantObjects: 21
[Line 29] gdf_tree_circles.shape: (378, 1)
x0 =  -66983.49795860096
y0 =  6798562.116878612
draw_grid_box : dx =  5
draw_grid_box : dy =  4
📍 Center of box: (51.990025, -0.599702)
x0 =  -68390.01972477396
y0 =  6800232.67711292
draw_grid_box : dx =  5
draw_grid_box : dy =  4
📍 Center of box: (52.000395, -0.614877)
x0 =  -68390.01972477396
y0 =  6800232.67711292
draw_grid_box : dx =  5
draw_grid_box : dy =  4
📍 Center of box: (52.000385, -0.614572)
x0 =  -68512.47116464656
y0 =  6800124.188707629
draw_grid_box : dx =  5
draw_grid_box : dy =  4
📍 Center of box: (51.999732, -0.616028)
Constants.n_cols =  21
Constants.n_rows =  18
x0 =  -68390.01972477396
y0 =  6799654.0890871575
draw_grid_box : dx =  5
draw_grid_box : dy =  4
📍 Center of box: (51.999782, -0.608699)
x0 =  -68796.33586616941
y0 =  6799925.297074551
draw_grid_box : dx =  5
draw_grid_box : dy =  4
📍 Center of box: 

In [2]:
#dates = Utils.generate_date_range("2025-01-25", "2025-01-26",  "%Y-%m-%d", 5)

#NDMI = Normalized Difference Moisture Index
# Output lists
ndmi_values = []
valid_dates = []

output_tif_cumulative = "ndmi_cumulative.tif"
output_tif = ""
gndvi_tif = ""
rgb_tif = ""

mgrs_tile = Utils.latlon_to_s2_tile(
    Constants.woburn_lat0,
    Constants.woburn_lon0
)

print(
    "RUN LOCATION:",
    Constants.woburn_lat0,
    Constants.woburn_lon0,
    "tile =", mgrs_tile
)

#assert mgrs_tile == "14QQG", (
#    f"Unexpected tile {mgrs_tile} "
#    f"for lat={Constants.woburn_lat0}, lon={Constants.woburn_lon0}"
#)
print("Done with cell!")

[Line 506] ... Mapped latitude  52.000593 , longitude  -0.614358  to mgrs_tile =  30UXC
RUN LOCATION: 52.000593 -0.614358 tile = 30UXC
Done with cell!


In [4]:
# Re-run the previous code after reset
#veracruz good dates
#2024-01-13
#2025-01-12
#Some Sentinel2 overflight dates in woburn are 2021-06-05, 2023-06-05 2025-06-04


#dates21 = Utils.generate_date_range("2021-03-04", "2021-06-30", Constants.woburn_lat0, Constants.woburn_lon0, "%Y-%m-%d")
#dates22 = Utils.generate_date_range("2022-03-04", "2022-06-30", Constants.woburn_lat0, Constants.woburn_lon0, "%Y-%m-%d")
dates23 = Utils.generate_date_range("2023-06-05", "2023-08-31", Constants.woburn_lat0, Constants.woburn_lon0, "%Y-%m-%d")
dates24 = Utils.generate_date_range("2024-06-04", "2024-08-31", Constants.woburn_lat0, Constants.woburn_lon0, "%Y-%m-%d")
dates25 = Utils.generate_date_range("2025-06-04", "2025-08-31", Constants.woburn_lat0, Constants.woburn_lon0, "%Y-%m-%d")
dates26 = Utils.generate_date_range("2026-06-04", "2026-08-31", Constants.woburn_lat0, Constants.woburn_lon0, "%Y-%m-%d")
dates = dates23+dates24+dates25+dates26
print("dates = ",dates)


# Sample NDMI at the given point
for date in dates:
    print("processing date ",date)
    # Download images corresponding to bands 2,3,4,8,11.
    b02_path = Utils.download_band_dynamic(date, "B02",Constants.woburn_lat0,Constants.woburn_lon0)  # Blue
    b03_path = Utils.download_band_dynamic(date, "B03",Constants.woburn_lat0,Constants.woburn_lon0)  # Green
    b04_path = Utils.download_band_dynamic(date, "B04",Constants.woburn_lat0,Constants.woburn_lon0)  # Red
    b08_path = Utils.download_band_dynamic(date, "B08",Constants.woburn_lat0,Constants.woburn_lon0)
    print(f"[Cell Line {inspect.currentframe().f_lineno}] ...  ")
    b11_path = Utils.download_band_dynamic(date, "B11",Constants.woburn_lat0,Constants.woburn_lon0)
    print(f"Cell [Line {inspect.currentframe().f_lineno}] ...  ")
    #b04_path = Utils.download_band_dynamic(date, "B04",Constants.woburn_lat0,Constants.woburn_lon0)
    
    print("set b02_path ",  b02_path)
    print("set b03_path ",  b03_path)
    print("set b04_path ",  b04_path)
    print("set b08_path ",  b08_path)
    print(f"Cell [Line {inspect.currentframe().f_lineno}] ...  ")
    print("set b11_path ",  b11_path)

    
    print(f"Cell [Line {inspect.currentframe().f_lineno}] ... About to try ")
    try:
        with rasterio.open(b08_path) as src_b08, rasterio.open(b11_path) as src_b11, rasterio.open(b03_path) as src_b03 , rasterio.open(b04_path) as src_b04 : # , rasterio.open(b02_path) as src_b02, rasterio.open(b03_path) as src_b03, rasterio.open(b04_path) as src_b04:
            print(f"Cell [Line {inspect.currentframe().f_lineno}] ... ")

            b11_data = src_b11.read(1)
            print(f"Cell [Line {inspect.currentframe().f_lineno}] ... ")

            b03_data = src_b03.read(1)
            b04_data = src_b04.read(1)
            print(f"Cell [Line {inspect.currentframe().f_lineno}] ... ")

            resampled_b11 = np.empty((10980, 10980), dtype="float32")
            # Create an ndmi tif for this date
            output_tif = Utils.make_path_name("s2_derived_tifs", mgrs_tile,date, "ndmi","tif")
            os.path.exists(output_tif) or (Utils.write_ndmi_geotiff(  b08_path, b11_path, output_tif), print("✅ NDMI GeoTIFF in UTM saved to:", output_tif))
            out_dtype = "uint8"
            print(f"Cell [Line {inspect.currentframe().f_lineno}] ... ")

            rgb_tif    = Utils.make_path_name("s2_derived_tifs", mgrs_tile,date, "rgb."+out_dtype,"tif")
            os.path.exists(rgb_tif) or (Utils.write_rgb_geotiff_3857(b02_path, b03_path, b04_path, rgb_tif, out_dtype), print("✅ RGB GeoTIFF in UTM saved to:", rgb_tif))
            #out_dtype = "float32"
            #rgb_tif    = Utils.make_path_name("s2_derived_tifs", mgrs_tile,date, "rgb."+out_dtype,"tif")
            #os.path.exists(rgb_tif) or (Utils.write_rgb_geotiff_3857(b02_path, b03_path, b04_path, rgb_tif, out_dtype), print("✅ RGB GeoTIFF in UTM saved to:", rgb_tif) )    
            #out_dtype = "float32"
            print(f"Cell [Line {inspect.currentframe().f_lineno}] ... ")

            gndvi_tif = Utils.make_path_name("s2_derived_tifs", mgrs_tile,date, "gndvi","tif")
            print(f"Cell [Line {inspect.currentframe().f_lineno}] ... ")
            os.path.exists(gndvi_tif) or (Utils.write_gndvi_geotiff(  b08_path, b03_path, gndvi_tif), print("✅ GNDVI GeoTIFF in UTM saved to:", gndvi_tif))
            print(f"Cell [Line {inspect.currentframe().f_lineno}] ... ")
                
    except Exception as e:
        print(f"Skipping {date} due to error: {e}")

print("output_tif =", repr(output_tif))
print("exists =", os.path.exists(output_tif) if output_tif else False)

print(f"Cell [Line {inspect.currentframe().f_lineno}] ... Done with cell! ")


[Line 506] ... Mapped latitude  52.000593 , longitude  -0.614358  to mgrs_tile =  30UXC
First acquisition: 2023-06-05
[Line 506] ... Mapped latitude  52.000593 , longitude  -0.614358  to mgrs_tile =  30UXC
First acquisition: 2024-06-04
[Line 506] ... Mapped latitude  52.000593 , longitude  -0.614358  to mgrs_tile =  30UXC
First acquisition: 2025-06-04
[Line 506] ... Mapped latitude  52.000593 , longitude  -0.614358  to mgrs_tile =  30UXC
First acquisition: 2026-06-04
dates =  ['2023-06-05', '2023-06-10', '2023-06-15', '2023-06-20', '2023-06-25', '2023-06-30', '2023-07-05', '2023-07-10', '2023-07-15', '2023-07-20', '2023-07-25', '2023-07-30', '2023-08-04', '2023-08-09', '2023-08-14', '2023-08-19', '2023-08-24', '2023-08-29', '2024-06-04', '2024-06-09', '2024-06-14', '2024-06-19', '2024-06-24', '2024-06-29', '2024-07-04', '2024-07-09', '2024-07-14', '2024-07-19', '2024-07-24', '2024-07-29', '2024-08-03', '2024-08-08', '2024-08-13', '2024-08-18', '2024-08-23', '2024-08-28', '2025-06-04', 

/var/folders/dm/6jp_xjys25gdd27bjkbvnhk80000gn/T/ipykernel_24343/646155064.py:44: DeprecationWarning: Setting the shape on a NumPy array has been deprecated in NumPy 2.5.
As an alternative, you can create a new view using np.reshape (with copy=False if needed).
  b11_data = src_b11.read(1)


Cell [Line 45] ... 


/var/folders/dm/6jp_xjys25gdd27bjkbvnhk80000gn/T/ipykernel_24343/646155064.py:47: DeprecationWarning: Setting the shape on a NumPy array has been deprecated in NumPy 2.5.
As an alternative, you can create a new view using np.reshape (with copy=False if needed).
  b03_data = src_b03.read(1)
/var/folders/dm/6jp_xjys25gdd27bjkbvnhk80000gn/T/ipykernel_24343/646155064.py:48: DeprecationWarning: Setting the shape on a NumPy array has been deprecated in NumPy 2.5.
As an alternative, you can create a new view using np.reshape (with copy=False if needed).
  b04_data = src_b04.read(1)


Cell [Line 49] ... 
band = > ndmi <
extension = > tif <
✅ NDMI GeoTIFF in UTM saved to: s2_derived_tifs/30UXC_2023-06-05_ndmi.tif
Cell [Line 56] ... 
band = > rgb.uint8 <
extension = > tif <
[Line 1284] ...       
[Line 1312] ...       
[Line 1333] ...       
[Line 1369] ...       
[Line 1374] ...       
[Line 1410] ... About to write  > s2_derived_tifs/30UXC_2023-06-05_rgb.uint8.tif <
[Line 1413] ... Exiting sub      
✅ RGB GeoTIFF in UTM saved to: s2_derived_tifs/30UXC_2023-06-05_rgb.uint8.tif
Cell [Line 64] ... 
band = > gndvi <
extension = > tif <
Cell [Line 67] ... 
[Line 1178] ... 
✅ GNDVI GeoTIFF in UTM saved to: s2_derived_tifs/30UXC_2023-06-05_gndvi.tif
Cell [Line 69] ... 
processing date  2023-06-10
[Line 506] ... Mapped latitude  52.000593 , longitude  -0.614358  to mgrs_tile =  30UXC
[Line 506] ... Mapped latitude  52.000593 , longitude  -0.614358  to mgrs_tile =  30UXC
[Line 506] ... Mapped latitude  52.000593 , longitude  -0.614358  to mgrs_tile =  30UXC
[Line 506] ... Ma

...

Cell [Line 49] ... 
band = > ndmi <
extension = > tif <
✅ NDMI GeoTIFF in UTM saved to: s2_derived_tifs/30UXC_2026-08-23_ndmi.tif
Cell [Line 56] ... 
band = > rgb.uint8 <
extension = > tif <
[Line 1284] ...       
[Line 1312] ...       
[Line 1333] ...       
[Line 1369] ...       
[Line 1374] ...       
[Line 1410] ... About to write  > s2_derived_tifs/30UXC_2026-08-23_rgb.uint8.tif <
[Line 1413] ... Exiting sub      
✅ RGB GeoTIFF in UTM saved to: s2_derived_tifs/30UXC_2026-08-23_rgb.uint8.tif
Cell [Line 64] ... 
band = > gndvi <
extension = > tif <
Cell [Line 67] ... 
[Line 1178] ... 
✅ GNDVI GeoTIFF in UTM saved to: s2_derived_tifs/30UXC_2026-08-23_gndvi.tif
Cell [Line 69] ... 
processing date  2026-08-28
[Line 506] ... Mapped latitude  52.000593 , longitude  -0.614358  to mgrs_tile =  30UXC
[Line 506] ... Mapped latitude  52.000593 , longitude  -0.614358  to mgrs_tile =  30UXC
[Line 506] ... Mapped latitude  52.000593 , longitude  -0.614358  to mgrs_tile =  30UXC
[Line 506] ... Ma

/var/folders/dm/6jp_xjys25gdd27bjkbvnhk80000gn/T/ipykernel_24343/646155064.py:44: DeprecationWarning: Setting the shape on a NumPy array has been deprecated in NumPy 2.5.
As an alternative, you can create a new view using np.reshape (with copy=False if needed).
  b11_data = src_b11.read(1)


Cell [Line 45] ... 


/var/folders/dm/6jp_xjys25gdd27bjkbvnhk80000gn/T/ipykernel_24343/646155064.py:47: DeprecationWarning: Setting the shape on a NumPy array has been deprecated in NumPy 2.5.
As an alternative, you can create a new view using np.reshape (with copy=False if needed).
  b03_data = src_b03.read(1)
/var/folders/dm/6jp_xjys25gdd27bjkbvnhk80000gn/T/ipykernel_24343/646155064.py:48: DeprecationWarning: Setting the shape on a NumPy array has been deprecated in NumPy 2.5.
As an alternative, you can create a new view using np.reshape (with copy=False if needed).
  b04_data = src_b04.read(1)


Cell [Line 49] ... 
band = > ndmi <
extension = > tif <
✅ NDMI GeoTIFF in UTM saved to: s2_derived_tifs/30UXC_2026-08-28_ndmi.tif
Cell [Line 56] ... 
band = > rgb.uint8 <
extension = > tif <
[Line 1284] ...       
[Line 1312] ...       
[Line 1333] ...       
[Line 1369] ...       
[Line 1374] ...       
[Line 1410] ... About to write  > s2_derived_tifs/30UXC_2026-08-28_rgb.uint8.tif <
[Line 1413] ... Exiting sub      
✅ RGB GeoTIFF in UTM saved to: s2_derived_tifs/30UXC_2026-08-28_rgb.uint8.tif
Cell [Line 64] ... 
band = > gndvi <
extension = > tif <
Cell [Line 67] ... 
[Line 1178] ... 
✅ GNDVI GeoTIFF in UTM saved to: s2_derived_tifs/30UXC_2026-08-28_gndvi.tif
Cell [Line 69] ... 
output_tif = 's2_derived_tifs/30UXC_2026-08-28_ndmi.tif'
exists = True
Cell [Line 77] ... Done with cell! 


In [7]:
import leafmap
importlib.reload(Constants)
importlib.reload(ConstantObjects)
print(f"[Line {inspect.currentframe().f_lineno}] ... lat0 = ",Constants.woburn_lat0)
print(f"[Line {inspect.currentframe().f_lineno}] ... lon0 = ",Constants.woburn_lon0)

m2 = leafmap.Map(basemap="Esri.WorldImagery")
#m2 = leafmap.Map(center=(Constants.woburn_lat0,Constants.woburn_lon0), zoom=15)
#m2.add_raster(output_tif, layer_name="NDMI", opacity=0.9, nodata=float("nan"))

m2.set_center(Constants.woburn_lon0, Constants.woburn_lat0,  zoom=15)

            
m2.add_gdf(ConstantObjects.gdf_box_woburn_soil_mine, layer_name="Woburn soil mine")
m2.add_gdf(ConstantObjects.gdf_box_woburn_abbey_park, layer_name="Woburn Abbey Park")
m2.add_gdf(ConstantObjects.gdf_box_willow, layer_name="Willow")
for i in range(25) :
    m2.add_gdf(ConstantObjects.gdf_box_woburn_soil_mine_20x20(i,0),  layer_name="Woburn soil mine 0,0")
m2

[Line 14] n_cols in ConstantObjects: 21
[Line 16] n_cols: 21, n_rows: 18
[Line 27] n_cols in ConstantObjects: 21
[Line 29] gdf_tree_circles.shape: (378, 1)
x0 =  -66983.49795860096
y0 =  6798562.116878612
draw_grid_box : dx =  5
draw_grid_box : dy =  4
📍 Center of box: (51.990025, -0.599702)
x0 =  -68390.01972477396
y0 =  6800232.67711292
draw_grid_box : dx =  5
draw_grid_box : dy =  4
📍 Center of box: (52.000395, -0.614877)
x0 =  -68390.01972477396
y0 =  6800232.67711292
draw_grid_box : dx =  5
draw_grid_box : dy =  4
📍 Center of box: (52.000385, -0.614572)
x0 =  -68512.47116464656
y0 =  6800124.188707629
draw_grid_box : dx =  5
draw_grid_box : dy =  4
📍 Center of box: (51.999732, -0.616028)
Constants.n_cols =  21
Constants.n_rows =  18
x0 =  -68390.01972477396
y0 =  6799654.0890871575
draw_grid_box : dx =  5
draw_grid_box : dy =  4
📍 Center of box: (51.999782, -0.608699)
x0 =  -68796.33586616941
y0 =  6799925.297074551
draw_grid_box : dx =  5
draw_grid_box : dy =  4
📍 Center of box: 

Map(center=[52.000593, -0.614358], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title',…